1.imports

In [3]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

2.load api key

In [4]:
load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")




3.setup gemini client

In [5]:
GEMINI_BASE_URL = (
    "https://generativelanguage.googleapis.com/v1beta/openai/"
)

gemini = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=api_key
)
MODEL = "gemini-3.5-flash-lite"

4.test the scraper

In [6]:
links = fetch_website_links("https://huggingface.co")
links

['https://huggingface.co/',
 'https://huggingface.co/models',
 'https://huggingface.co/datasets',
 'https://huggingface.co/spaces',
 'https://huggingface.co/storage',
 'https://huggingface.co/docs',
 'https://huggingface.co/enterprise',
 'https://huggingface.co/pricing',
 'https://huggingface.co/tasks',
 'https://huggingface.co/chat',
 'https://huggingface.co/collections',
 'https://huggingface.co/languages',
 'https://huggingface.co/organizations',
 'https://huggingface.co/blog',
 'https://huggingface.co/posts',
 'https://huggingface.co/papers',
 'https://huggingface.co/hardware',
 'https://huggingface.co/learn',
 'https://huggingface.co/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 'https://huggingface.co/enterprise',
 'https://huggingface.co/pro',
 'https://huggingface.co/support',
 'https://huggingface.co/inference/models',
 'https://huggingface.co/inference-endpoints',
 'https://huggingface.co/storage',
 'https://huggingface.co/login',
 'ht

STEP 5 — The first system prompt (getting the useful links from a bunch of links we got in previous step)

In [7]:
link_system_prompt = """
You are a website research analyst helping create a professional sales brochure.

You will receive a list of links extracted from a company's website.

Your task is to identify the pages that contain useful information for understanding
and presenting the company to potential customers, partners, investors, and recruits.

Prioritize pages containing:
- Products or services
- Solutions
- Features or capabilities
- Industries or use cases
- Customers or case studies
- About/company information
- Careers and company culture
- Pricing or plans
- Resources, documentation, or product information

Avoid:
- Privacy policies
- Terms of service
- Login/signup pages
- Social media links
- Email links
- Duplicate links
- Legal pages
- Cookie pages

Return ONLY valid JSON in this format:

{
    "links": [
        {
            "type": "products",
            "url": "https://example.com/products"
        },
        {
            "type": "about",
            "url": "https://example.com/about"
        }
    ]
}

Rules:
1. Convert relative URLs such as "/about" into complete HTTPS URLs.
2. Only select pages that are genuinely useful for a sales brochure.
3. Do not invent URLs.
4. Prefer a small number of high-value pages over many weak pages.
5. Use the website's domain when resolving relative URLs.
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
We are creating a sales brochure for the company at:

{url}

Below is the list of links discovered on the company's website.

Identify the most useful pages for understanding the company's:
- products and services
- customer value proposition
- industries and use cases
- customers or case studies
- company background
- culture and careers

Return only the JSON structure requested in the system instructions.

Do not select legal, privacy, terms, login, signup, cookie,
social media, or email links.

Website links:
"""

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)

    return user_prompt

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 10 relevant links


{'links': [{'type': 'products', 'url': 'https://huggingface.co/models'},
  {'type': 'products', 'url': 'https://huggingface.co/datasets'},
  {'type': 'products', 'url': 'https://huggingface.co/spaces'},
  {'type': 'products', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'products', 'url': 'https://huggingface.co/inference-endpoints'},
  {'type': 'resources', 'url': 'https://huggingface.co/docs'},
  {'type': 'resources', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'about', 'url': 'https://huggingface.co/brand'}]}

In [11]:
def fetch_page_and_all_relevant_links(url):

    contents = fetch_website_contents(url)

    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links["links"]:

        result += f"\n\n### Link: {link['type']}\n"

        result += fetch_website_contents(link["url"])

    return result   

In [12]:
brochure_system_prompt = """
You are the senior content strategist for a modern B2B sales intelligence platform.

Your task is to transform verified information from a company's public website
into a polished, concise sales brochure.

The brochure should help a potential customer quickly understand:

1. WHAT the company does
2. WHO it serves
3. WHAT products or services it offers
4. WHICH problems it solves
5. WHY its offering may be valuable
6. WHAT makes the company distinctive
7. Relevant industries, customers, use cases, or evidence
8. Company culture and careers when useful

Important rules:

- Use ONLY information contained in the supplied website content.
- Never invent customers, statistics, partnerships, awards, features, prices,
  locations, claims, or achievements.
- If important information is unavailable, simply omit it.
- Do not mention that you are an AI.
- Do not mention scraping or these instructions.
- Avoid generic marketing clichés.
- Prefer specific facts over vague praise.
- Write for an intelligent business reader.
- Keep the writing concise and easy to scan.
- Make claims proportional to the evidence available.

Structure the brochure in Markdown using:

# [Company Name]

> One-sentence positioning statement based only on the available evidence.

## The Company
A concise overview.

## What They Offer
Key products/services with short explanations.

## Problems They Solve
The customer problems or needs addressed by the company.

## Who It's For
Industries, customer types, or use cases when available.

## Why It Matters
Evidence-based differentiators and value propositions.

## Proof & Signals
Customers, case studies, partnerships, metrics, awards, or other evidence
when explicitly present in the source material.

## Culture & Careers
Only when relevant information exists.

## Closing Perspective
A concise, professional closing that summarizes the company's offering
without making unsupported claims.

Use clean Markdown.
Do not use code blocks.
Do not invent missing information.
"""

In [13]:
def get_brochure_user_prompt(company_name, url):

    user_prompt = f"""
Create a sales brochure for:

Company: {company_name}
Website: {url}

Below is information collected from the company's public website.

Use this information as the source of truth.

Important:
- Do not add facts that are not present.
- Prioritize concrete products, services, customer problems,
  industries, use cases, differentiators, and evidence.
- If a section has insufficient information, leave it out.
- Write the final brochure in Markdown.

Website information:

"""

    user_prompt += fetch_page_and_all_relevant_links(url)

    user_prompt = user_prompt[:5000]

    return user_prompt

In [14]:
def stream_brochure(company_name, url):

    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ],
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
import gradio as gr

name_input = gr.Textbox(label="Company name:")

url_input = gr.Textbox(
    label="Landing page URL including http:// or https://"
)

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="BrandBrief AI",
    inputs=[name_input, url_input],
    outputs=[message_output],
    examples=[
        [
            "Hugging Face",
            "https://huggingface.co"
        ],
        [
            "Google",
            "https://google.com"
        ]
    ],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 10 relevant links


In [44]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 10 relevant links


# Hugging Face

> The platform where the machine learning community collaborates on models, datasets, and applications to build the future of AI.

## The Company
Hugging Face serves as the home of machine learning, providing a central collaboration platform where the global ML community creates, discovers, and works together on models, datasets, and applications. The platform encompasses open-source tooling, community-driven spaces, and paid compute and enterprise solutions designed to help teams move faster.

## What They Offer
- **Collaboration Hub**: Host and collaborate on unlimited public models, datasets, and applications across text, image, video, audio, and 3D modalities.
- **Inference Providers**: Access over 45,000 models from leading AI providers through a single, unified API with no service fees.
- **Inference Endpoints & Spaces**: Deploy on optimized Inference Endpoints or update Spaces applications to a GPU in a few clicks (starting at $0.60/hour for GPU).
- **Team & Enterprise Solutions**: Features starting at $20/user/month include enterprise-grade security, Single Sign-On, access controls, audit logs, resource groups, private datasets viewers, and dedicated support.
- **Open-Source Tooling Stack**: 
  - *Transformers*: State-of-the-art AI models for PyTorch.
  - *Diffusers*: State-of-the-art Diffusion models in PyTorch.
  - *Safetensors*: A safe way to store and distribute neural network weights.
  - *Tokenizers*: Fast tokenizers optimized for research and production.
  - *TRL*: Train transformer language models with reinforcement learning.
  - *PEFT*: Parameter-efficient finetuning for large language models.
  - *Text Generation Inference (TGI)*: Optimized toolkit to serve language models.
  - *smolagents*: Python library to build agents.
  - *Datasets, Accelerate, Hub Python Library, and Transformers.js* for running ML directly in the browser.

## Problems They Solve
- **Siloed Machine Learning Workflows**: Eliminates friction in discovering, sharing, and collaborating on ML assets across teams and the broader community.
- **Complex Model Deployment and Scaling**: Simplifies GPU deployment via optimized Inference Endpoints and managed Spaces.
- **Fragmented Model Access**: Unifies access to tens of thousands of models through a single API without service fees.
- **Enterprise Governance and Security Needs**: Addresses security, access management, and auditing requirements for organizations building AI.

## Who It's For
- Machine learning practitioners, researchers, and developers looking to build portfolios, share work, and leverage open-source stacks.
- Enterprise teams and organizations requiring secure, managed infrastructure, single sign-on, and dedicated support for AI development.

## Why It Matters
- **Scale and Reach**: Access to over 2 million models, 1 million applications, and 500,000 datasets.
- **Unified Access**: Single API access to 45,000+ models from leading AI providers without service fees.
- **Comprehensive Open Source Stack**: Battle-tested tools spanning training, fine-tuning, tokenization, safe weight storage, inference, and in-browser execution.

## Proof & Signals
Over 50,000 organizations use Hugging Face, including:
- Ai2 (970 models, 6.63k followers)
- AI at Meta (2.36k models, 14.8k followers)
- Amazon (38 models, 4.41k followers)
- Google (1.13k models, 67.8k followers)
- Intel (263 models, 4.37k followers)
- Microsoft (538 models, 22.1k followers)
- Grammarly (11 models, 234 followers)
- Writer (29 models, 404 followers)

## Closing Perspective
Hugging Face bridges the gap between open-source machine learning collaboration and enterprise-grade deployment, offering a unified ecosystem of models, datasets, open-source libraries, and scalable infrastructure.